In [1]:
import h5py
import pandas as pd
import numpy as np

# Load file
f = h5py.File("datasets/metr-la.h5", "r")

# Read speed values and sensor IDs
values = f['df']['block0_values'][:]       # speeds (34272 × 207)
axis0_raw = f['df']['axis0'][:]            # sensor IDs

# Decode column names
sensor_ids = [x.decode() for x in axis0_raw]

# Generate timestamps (5-min interval)
num_rows = values.shape[0]
start = pd.Timestamp("2012-03-01 00:00:00")
timestamps = start + pd.to_timedelta(np.arange(num_rows) * 5, unit='min')

# Create dataframe
df = pd.DataFrame(values, columns=sensor_ids)
df.insert(0, "timestamp", timestamps)

print(df.head())
print(df.shape)


            timestamp     773869     767541     767542     717447     717446  \
0 2012-03-01 00:00:00  64.375000  67.625000  67.125000  61.500000  66.875000   
1 2012-03-01 00:05:00  62.666667  68.555556  65.444444  62.444444  64.444444   
2 2012-03-01 00:10:00  64.000000  63.750000  60.000000  59.000000  66.500000   
3 2012-03-01 00:15:00   0.000000   0.000000   0.000000   0.000000   0.000000   
4 2012-03-01 00:20:00   0.000000   0.000000   0.000000   0.000000   0.000000   

      717445  773062  767620     737529  ...     772167  769372     774204  \
0  68.750000  65.125  67.125  59.625000  ...  45.625000  65.500  64.500000   
1  68.111111  65.000  65.000  57.444444  ...  50.666667  69.875  66.666667   
2  66.250000  64.500  64.250  63.875000  ...  44.125000  69.000  56.500000   
3   0.000000   0.000   0.000   0.000000  ...   0.000000   0.000   0.000000   
4   0.000000   0.000   0.000   0.000000  ...   0.000000   0.000   0.000000   

      769806  717590     717592     717595     772

In [2]:
# Melt to (timestamp, sensor_id, speed)
df_long = df.melt(id_vars="timestamp", var_name="sensor_id", value_name="speed")

# Add features
df_long["hour"] = df_long["timestamp"].dt.hour
df_long["dayofweek"] = df_long["timestamp"].dt.dayofweek
df_long["is_weekend"] = (df_long["dayofweek"] >= 5).astype(int)
df_long = df_long.sort_values(["sensor_id", "timestamp"])

# Lag features
df_long["lag1"] = df_long.groupby("sensor_id")["speed"].shift(1)
df_long["lag3"] = df_long.groupby("sensor_id")["speed"].shift(3)
df_long["lag6"] = df_long.groupby("sensor_id")["speed"].shift(6)

# Drop NA rows
df_long = df_long.dropna()

print(df_long.head())
print(df_long.shape)


                  timestamp sensor_id      speed  hour  dayofweek  is_weekend  \
1130982 2012-03-01 00:30:00    716328  65.250000     0          3           0   
1130983 2012-03-01 00:35:00    716328  63.000000     0          3           0   
1130984 2012-03-01 00:40:00    716328  55.875000     0          3           0   
1130985 2012-03-01 00:45:00    716328  65.750000     0          3           0   
1130986 2012-03-01 00:50:00    716328  66.777778     0          3           0   

              lag1       lag3       lag6  
1130982  65.666667   0.000000  66.875000  
1130983  65.250000   0.000000  67.444444  
1130984  63.000000  65.666667  65.000000  
1130985  55.875000  65.250000   0.000000  
1130986  65.750000  63.000000   0.000000  
(7093062, 9)


In [9]:
import gc

features = ["sensor_id_encoded","hour","dayofweek","is_weekend","lag1","lag3","lag6"]

for col in features:
    df_long[col] = df_long[col].astype('float32')
    gc.collect()

X = df_long[features]


In [11]:
# ============================================
#   FULL ML MODEL IN ONE CELL (OPTIMAL)
# ============================================

import pandas as pd
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib

# --------------------------------------------
# 1) LOAD + MEMORY REDUCE
# --------------------------------------------
df = df_long   # your dataset already loaded

features = ["sensor_id_encoded","hour","dayofweek","is_weekend","lag1","lag3","lag6"]

# reduce memory
for col in features:
    df[col] = pd.to_numeric(df[col], downcast='float')
df["speed"] = pd.to_numeric(df["speed"], downcast='float')
gc.collect()

# --------------------------------------------
# 2) PREPARE X, y
# --------------------------------------------
X = df[features]
y = df["speed"]

# --------------------------------------------
# 3) TRAIN / TEST SPLIT (IMPORTANT!)
# --------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# --------------------------------------------
# 4) TRAIN MODEL (XGBoost)
# --------------------------------------------
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='reg:squarederror',
    tree_method='hist'
)

model.fit(X_train, y_train)

# --------------------------------------------
# 5) EVALUATE
# --------------------------------------------
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)

# --------------------------------------------
# 6) SAVE MODEL
# --------------------------------------------
joblib.dump(model, "traffic_speed_model.pkl")
print("Model saved as traffic_speed_model.pkl")

# ============================================


MAE: 2.8132684230804443
Model saved as traffic_speed_model.pkl
